# Transformer Research Notebook

This notebook is intentionally monolithic again.

The transformer workflow lives here end to end:
- FineWeb Edu loading
- repo-local specialization corpus loading
- phase corpus construction
- tokenizer training / loading
- token export and train/val splits
- modern dense / optional MoE model definition
- launch bundle generation for distributed training
- dashboards, probes, and post-training utilities

The helper files can still exist on disk, but this notebook is meant to be the main debugging and experimentation surface.

In [ ]:
# Environment, runtime checks, and top-level paths

from pathlib import Path
import json
import math
import random
import tempfile

import numpy as np
import plotly.io as pio
import plotly.graph_objects as go
import sympy as sp
import ipywidgets as widgets
from IPython.display import Markdown, display

import torch
import torch.nn.functional as F

try:
    import datasets
    print("datasets version:", datasets.__version__)
except ImportError:
    datasets = None
    print("Install with: %pip install datasets")

try:
    import sentencepiece
    print("sentencepiece available")
except ImportError:
    sentencepiece = None
    print("Install with: %pip install sentencepiece")

PROJECT_ROOT = Path.cwd()
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts_transformer"
ARTIFACT_ROOT.mkdir(exist_ok=True)
CORPUS_ROOT = ARTIFACT_ROOT / "corpus"
CORPUS_ROOT.mkdir(exist_ok=True)
TOKENIZER_ROOT = ARTIFACT_ROOT / "tokenizer"
TOKENIZER_ROOT.mkdir(exist_ok=True)
PHASE_TOKEN_ROOT = ARTIFACT_ROOT / "phase_tokens"
PHASE_TOKEN_ROOT.mkdir(exist_ok=True)
LAUNCH_ROOT = ARTIFACT_ROOT / "launch"
LAUNCH_ROOT.mkdir(exist_ok=True)

pio.renderers.default = "notebook_connected"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Project root:", PROJECT_ROOT)
print("Artifact root:", ARTIFACT_ROOT)

In [ ]:
# Review docs for orientation

for rel in ['transformer_review.md', 'transformer_experiment_plan.md']:
    path = PROJECT_ROOT / rel
    if path.exists():
        display(Markdown(path.read_text(encoding='utf-8')))

In [ ]:
# Data and corpus helpers

from __future__ import annotations

import json
import re
from dataclasses import asdict, dataclass, field
from hashlib import sha1
from pathlib import Path
from typing import Iterable

try:
    from datasets import load_dataset
except ImportError:  # pragma: no cover - notebook runtime may install it later
    load_dataset = None


@dataclass
class TokenizerRecipe:
    vocab_size: int = 32000
    model_type: str = "unigram"
    character_coverage: float = 1.0
    sample_max_lines: int = 800000
    sample_stride: int = 8


@dataclass
class FineWebEduRecipe:
    dataset_name: str = "HuggingFaceFW/fineweb-edu"
    split: str = "train"
    streaming: bool = True
    max_docs: int = 50000
    min_words: int = 40
    max_chars_per_doc: int = 12000
    text_keys: tuple[str, ...] = ("text", "content", "raw_content", "document")
    dedupe_on_text: bool = True


@dataclass
class PhaseRecipe:
    name: str
    broad_web_weight: int
    repo_weight: int
    reasoning_weight: int
    science_weight: int
    code_weight: int
    max_chars_per_doc: int = 12000
    chunk_words: int = 220


@dataclass
class EvalSlice:
    name: str
    description: str
    source_labels: list[str]


@dataclass
class CorpusRecipe:
    tokenizer: TokenizerRecipe = field(default_factory=TokenizerRecipe)
    fineweb_edu: FineWebEduRecipe = field(default_factory=FineWebEduRecipe)
    phases: list[PhaseRecipe] = field(
        default_factory=lambda: [
            PhaseRecipe("tokenizer", broad_web_weight=92, repo_weight=3, reasoning_weight=1, science_weight=2, code_weight=2),
            PhaseRecipe("base", broad_web_weight=96, repo_weight=1, reasoning_weight=1, science_weight=1, code_weight=1),
            PhaseRecipe("adapt", broad_web_weight=65, repo_weight=10, reasoning_weight=8, science_weight=10, code_weight=7),
            PhaseRecipe("sft", broad_web_weight=0, repo_weight=35, reasoning_weight=35, science_weight=20, code_weight=10),
        ]
    )
    eval_slices: list[EvalSlice] = field(
        default_factory=lambda: [
            EvalSlice("broad_edu_holdout", "Held-out educational web text from FineWeb Edu", ["broad_web"]),
            EvalSlice("repo_science", "README, AGENTS, and scientific notebook markdown", ["repo_markdown", "science_notebook"]),
            EvalSlice("systems_lm", "Transformer/distributed systems text", ["transformer_notebook", "repo_markdown"]),
            EvalSlice("instruction", "Curated instruction/reasoning examples", ["reasoning_seed"]),
        ]
    )


def _normalize_whitespace(text: str) -> str:
    lines = [re.sub(r"\s+", " ", line).strip() for line in text.splitlines()]
    lines = [line for line in lines if line]
    return "\n".join(lines).strip()


def _chunk_words(text: str, chunk_words: int) -> list[str]:
    words = text.split()
    chunks = []
    for start in range(0, len(words), chunk_words):
        chunk = " ".join(words[start : start + chunk_words]).strip()
        if chunk:
            chunks.append(chunk)
    return chunks


def _stable_hash(text: str) -> str:
    return sha1(text.encode("utf-8")).hexdigest()


def _extract_text_from_row(row: dict, text_keys: Iterable[str]) -> str | None:
    for key in text_keys:
        value = row.get(key)
        if value is None:
            continue
        text = _normalize_whitespace(str(value))
        if text:
            return text
    return None


def extract_notebook_cells(path: Path) -> list[dict[str, str]]:
    notebook = json.loads(path.read_text(encoding="utf-8"))
    docs = []
    for idx, cell in enumerate(notebook.get("cells", [])):
        source = _normalize_whitespace("".join(cell.get("source", [])))
        if not source:
            continue
        label = "markdown" if cell.get("cell_type") == "markdown" else "code"
        docs.append({"id": f"{path.name}:{idx}", "label": label, "text": source})
    return docs


def build_repo_local_corpus(root: Path | str) -> list[dict[str, str]]:
    root = Path(root)
    documents: list[dict[str, str]] = []

    markdown_paths = [root / "README.md", root / "agents.md"]
    for path in markdown_paths:
        if path.exists():
            documents.append(
                {
                    "id": path.name,
                    "label": "repo_markdown",
                    "text": _normalize_whitespace(path.read_text(encoding="utf-8")),
                }
            )

    notebook_label_map = {
        "transformer.ipynb": "transformer_notebook",
        "scienceagentv1.ipynb": "science_notebook",
        "turbulence.ipynb": "science_notebook",
        "MHD_64.ipynb": "science_notebook",
    }
    for notebook_path in sorted(root.glob("*.ipynb")):
        label = notebook_label_map.get(notebook_path.name, "notebook")
        for cell in extract_notebook_cells(notebook_path):
            if cell["label"] == "markdown":
                documents.append({"id": cell["id"], "label": label, "text": cell["text"]})
            elif notebook_path.name == "transformer.ipynb":
                # Keep commented code and section headers from the transformer notebook as local systems text.
                if "#" in cell["text"]:
                    documents.append({"id": cell["id"], "label": label, "text": cell["text"]})

    reasoning_seed_prompts = [
        "Explain why grouped-query attention reduces KV-cache growth for a small decoder-only LM.",
        "Compare RMSNorm and LayerNorm for a student-scale autoregressive model.",
        "Describe how domain adaptation should differ from broad pretraining in this repository.",
        "Explain why a notebook-first training workflow can still support serious systems instrumentation.",
    ]
    for idx, prompt in enumerate(reasoning_seed_prompts):
        documents.append(
            {
                "id": f"reasoning_seed:{idx}",
                "label": "reasoning_seed",
                "text": prompt,
            }
        )

    deduped = {}
    for doc in documents:
        normalized = _normalize_whitespace(doc["text"])
        if len(normalized.split()) < 8:
            continue
        key = _stable_hash(normalized)
        deduped.setdefault(key, {"id": doc["id"], "label": doc["label"], "text": normalized})
    return list(deduped.values())


def stream_fineweb_edu(recipe: FineWebEduRecipe) -> list[dict[str, str]]:
    if load_dataset is None:
        raise ImportError(
            "The `datasets` library is required to load FineWeb Edu. "
            "Install it in the notebook environment with `%pip install datasets`."
        )

    dataset = load_dataset(recipe.dataset_name, split=recipe.split, streaming=recipe.streaming)
    documents: list[dict[str, str]] = []
    seen = set()
    kept = 0

    for idx, row in enumerate(dataset):
        text = _extract_text_from_row(row, recipe.text_keys)
        if text is None:
            continue
        text = text[: recipe.max_chars_per_doc].strip()
        if len(text.split()) < recipe.min_words:
            continue

        if recipe.dedupe_on_text:
            key = _stable_hash(text)
            if key in seen:
                continue
            seen.add(key)

        documents.append(
            {
                "id": f"fineweb_edu:{idx}",
                "label": "broad_web",
                "text": text,
                "source": recipe.dataset_name,
            }
        )
        kept += 1
        if kept >= recipe.max_docs:
            break

    return documents


def save_corpus_cache(documents: list[dict[str, str]], path: Path | str) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(documents, indent=2, ensure_ascii=False), encoding="utf-8")
    return path


def load_corpus_cache(path: Path | str) -> list[dict[str, str]]:
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8"))


def load_or_build_fineweb_edu(recipe: FineWebEduRecipe, cache_path: Path | str | None = None, force_refresh: bool = False) -> list[dict[str, str]]:
    if cache_path is not None:
        cache_path = Path(cache_path)
        if cache_path.exists() and not force_refresh:
            return load_corpus_cache(cache_path)

    documents = stream_fineweb_edu(recipe)
    if cache_path is not None:
        save_corpus_cache(documents, cache_path)
    return documents


def summarize_corpus_sources(documents: Iterable[dict[str, str]]) -> dict[str, dict[str, float]]:
    summary: dict[str, dict[str, float]] = {}
    for doc in documents:
        label = doc.get("label", "unknown")
        stats = summary.setdefault(label, {"documents": 0.0, "words": 0.0, "chars": 0.0})
        text = doc.get("text", "")
        stats["documents"] += 1.0
        stats["words"] += float(len(text.split()))
        stats["chars"] += float(len(text))
    return summary


def build_phase_documents(
    documents: Iterable[dict[str, str]],
    recipe: PhaseRecipe,
) -> dict[str, list[str]]:
    grouped = {
        "broad_web": [],
        "repo": [],
        "reasoning": [],
        "science": [],
        "code": [],
    }

    for doc in documents:
        text = doc["text"][: recipe.max_chars_per_doc]
        chunks = _chunk_words(text, recipe.chunk_words)
        label = doc["label"]
        if label == "broad_web":
            grouped["broad_web"].extend(chunks)
        elif label == "reasoning_seed":
            grouped["reasoning"].extend(chunks)
        elif label == "transformer_notebook":
            grouped["code"].extend(chunks)
        elif label == "science_notebook":
            grouped["science"].extend(chunks)
        elif label == "repo_markdown":
            grouped["repo"].extend(chunks)
        else:
            grouped["repo"].extend(chunks)
    return grouped


def weighted_phase_lines(phase_docs: dict[str, list[str]], recipe: PhaseRecipe) -> list[str]:
    def unique_lines(lines: list[str]) -> list[str]:
        out = []
        seen = set()
        for line in lines:
            normalized = _normalize_whitespace(line)
            if not normalized:
                continue
            key = _stable_hash(normalized)
            if key in seen:
                continue
            seen.add(key)
            out.append(normalized)
        return out

    broad_web = unique_lines(phase_docs["broad_web"])
    repo = unique_lines(phase_docs["repo"])
    reasoning = unique_lines(phase_docs["reasoning"])
    science = unique_lines(phase_docs["science"])
    code = unique_lines(phase_docs["code"])

    weighted = []
    weighted.extend(broad_web * max(recipe.broad_web_weight, 0))
    weighted.extend(repo * max(recipe.repo_weight, 0))
    weighted.extend(reasoning * max(recipe.reasoning_weight, 0))
    weighted.extend(science * max(recipe.science_weight, 0))
    weighted.extend(code * max(recipe.code_weight, 0))
    return weighted


def phase_summary(phase_name: str, phase_docs: dict[str, list[str]], phase_lines: list[str], recipe: PhaseRecipe) -> dict[str, object]:
    return {
        "phase_name": phase_name,
        "weights": asdict(recipe),
        "bucket_unique_counts": {
            "broad_web": len(set(phase_docs["broad_web"])),
            "repo": len(set(phase_docs["repo"])),
            "reasoning": len(set(phase_docs["reasoning"])),
            "science": len(set(phase_docs["science"])),
            "code": len(set(phase_docs["code"])),
        },
        "phase_line_count": len(phase_lines),
        "phase_word_count": sum(len(line.split()) for line in phase_lines),
    }


def build_corpus_manifest(
    broad_documents: list[dict[str, str]],
    repo_documents: list[dict[str, str]],
    recipe: CorpusRecipe,
    phase_line_map: dict[str, list[str]],
    phase_group_map: dict[str, dict[str, list[str]]],
    contamination: dict[str, dict[str, float]] | None = None,
) -> dict[str, object]:
    phase_meta = {}
    for phase in recipe.phases:
        if phase.name in phase_line_map and phase.name in phase_group_map:
            phase_meta[phase.name] = phase_summary(phase.name, phase_group_map[phase.name], phase_line_map[phase.name], phase)

    return {
        "fineweb_edu": asdict(recipe.fineweb_edu),
        "tokenizer": asdict(recipe.tokenizer),
        "source_summary": {
            "broad_web": summarize_corpus_sources(broad_documents),
            "repo_local": summarize_corpus_sources(repo_documents),
            "combined": summarize_corpus_sources([*broad_documents, *repo_documents]),
        },
        "phase_summary": phase_meta,
        "contamination": contamination or {},
    }


def contamination_report(train_lines: Iterable[str], eval_lines: Iterable[str], ngram_size: int = 8) -> dict[str, float]:
    def ngrams(text: str) -> set[str]:
        words = text.split()
        if len(words) < ngram_size:
            return {text}
        return {" ".join(words[i : i + ngram_size]) for i in range(len(words) - ngram_size + 1)}

    train_ngrams = set()
    for line in train_lines:
        train_ngrams.update(ngrams(line))

    eval_ngrams = set()
    for line in eval_lines:
        eval_ngrams.update(ngrams(line))

    overlap = train_ngrams & eval_ngrams
    overlap_rate = len(overlap) / max(len(eval_ngrams), 1)
    return {
        "train_ngram_count": float(len(train_ngrams)),
        "eval_ngram_count": float(len(eval_ngrams)),
        "overlap_ngram_count": float(len(overlap)),
        "overlap_rate": float(overlap_rate),
    }


def recipe_to_jsonable(recipe: CorpusRecipe) -> dict:
    return {
        "tokenizer": asdict(recipe.tokenizer),
        "fineweb_edu": asdict(recipe.fineweb_edu),
        "phases": [asdict(phase) for phase in recipe.phases],
        "eval_slices": [asdict(slc) for slc in recipe.eval_slices],
    }

In [ ]:
# Tokenization helpers

from __future__ import annotations

import json
from dataclasses import asdict
from pathlib import Path

import numpy as np


try:
    import sentencepiece as spm
except ImportError:  # pragma: no cover - installed in notebook runtime
    spm = None


def _require_sentencepiece():
    if spm is None:
        raise ImportError(
            "The `sentencepiece` library is required for tokenizer training. "
            "Install it in the notebook environment with `%pip install sentencepiece`."
        )


def sample_tokenizer_corpus(
    input_path: Path | str,
    output_path: Path | str,
    recipe: TokenizerRecipe,
    force_rebuild: bool = False,
) -> Path:
    input_path = Path(input_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not force_rebuild:
        return output_path

    kept = 0
    seen = 0
    with input_path.open("r", encoding="utf-8") as fin, output_path.open("w", encoding="utf-8") as fout:
        for line in fin:
            text = line.strip()
            if not text:
                continue
            if seen % max(recipe.sample_stride, 1) == 0:
                fout.write(text + "\n")
                kept += 1
                if kept >= recipe.sample_max_lines:
                    break
            seen += 1
    return output_path


def train_or_load_sentencepiece(
    training_text_path: Path | str,
    model_prefix: Path | str,
    recipe: TokenizerRecipe,
    force_retrain: bool = False,
) -> tuple[Path, Path]:
    _require_sentencepiece()
    model_prefix = Path(model_prefix)
    model_prefix.parent.mkdir(parents=True, exist_ok=True)
    model_path = model_prefix.with_suffix(".model")
    vocab_path = model_prefix.with_suffix(".vocab")

    if model_path.exists() and vocab_path.exists() and not force_retrain:
        return model_path, vocab_path

    spm.SentencePieceTrainer.Train(
        input=str(training_text_path),
        model_prefix=str(model_prefix),
        vocab_size=recipe.vocab_size,
        model_type=recipe.model_type,
        character_coverage=recipe.character_coverage,
        bos_id=1,
        eos_id=2,
        pad_id=0,
        unk_id=3,
        shuffle_input_sentence=True,
        input_sentence_size=min(recipe.sample_max_lines, 1_000_000),
        train_extremely_large_corpus=False,
    )
    return model_path, vocab_path


def load_sentencepiece_processor(model_path: Path | str):
    _require_sentencepiece()
    model_path = Path(model_path)
    processor = spm.SentencePieceProcessor()
    processor.load(str(model_path))
    return processor


def encode_text_file_to_tokens(
    input_path: Path | str,
    output_path: Path | str,
    processor,
    eos_id: int,
    force_rebuild: bool = False,
) -> tuple[Path, int]:
    input_path = Path(input_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not force_rebuild:
        arr = np.load(output_path, mmap_mode="r")
        return output_path, int(arr.shape[0])

    token_buffer: list[int] = []
    with input_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            text = line.strip()
            if not text:
                continue
            token_buffer.extend(processor.encode(text, out_type=int))
            token_buffer.append(int(eos_id))

    array = np.asarray(token_buffer, dtype=np.int32)
    np.save(output_path, array)
    return output_path, int(array.shape[0])


def split_token_array(
    token_path: Path | str,
    train_path: Path | str,
    val_path: Path | str,
    seq_len: int,
    val_fraction: float = 0.02,
    force_rebuild: bool = False,
) -> dict[str, int | str]:
    token_path = Path(token_path)
    train_path = Path(train_path)
    val_path = Path(val_path)
    train_path.parent.mkdir(parents=True, exist_ok=True)

    if train_path.exists() and val_path.exists() and not force_rebuild:
        train = np.load(train_path, mmap_mode="r")
        val = np.load(val_path, mmap_mode="r")
        return {
            "token_path": str(token_path),
            "train_path": str(train_path),
            "val_path": str(val_path),
            "total_tokens": int(train.shape[0] + val.shape[0]),
            "train_tokens": int(train.shape[0]),
            "val_tokens": int(val.shape[0]),
        }

    tokens = np.load(token_path, mmap_mode="r")
    total = int(tokens.shape[0])
    if total <= seq_len + 1:
        raise ValueError(f"Token stream too short for seq_len={seq_len}: {total}")

    split_idx = int((1.0 - val_fraction) * total)
    split_idx = max(split_idx, seq_len + 1)
    split_idx = min(split_idx, total - (seq_len + 1))

    train_tokens = np.asarray(tokens[:split_idx], dtype=np.int32)
    val_tokens = np.asarray(tokens[split_idx:], dtype=np.int32)
    np.save(train_path, train_tokens)
    np.save(val_path, val_tokens)

    return {
        "token_path": str(token_path),
        "train_path": str(train_path),
        "val_path": str(val_path),
        "total_tokens": int(total),
        "train_tokens": int(train_tokens.shape[0]),
        "val_tokens": int(val_tokens.shape[0]),
    }


def build_phase_token_artifacts(
    phase_file_map: dict[str, str | Path],
    tokenizer_dir: Path | str,
    token_dir: Path | str,
    recipe: TokenizerRecipe,
    seq_len: int,
    val_fraction: float = 0.02,
    force_retrain_tokenizer: bool = False,
    force_reencode: bool = False,
) -> dict[str, object]:
    tokenizer_dir = Path(tokenizer_dir)
    token_dir = Path(token_dir)
    tokenizer_dir.mkdir(parents=True, exist_ok=True)
    token_dir.mkdir(parents=True, exist_ok=True)

    tokenizer_phase_path = Path(phase_file_map["tokenizer"])
    sampled_corpus_path = tokenizer_dir / "tokenizer_sampled.txt"
    sampled_corpus_path = sample_tokenizer_corpus(tokenizer_phase_path, sampled_corpus_path, recipe, force_rebuild=force_retrain_tokenizer)

    model_prefix = tokenizer_dir / f"spm_{recipe.vocab_size}"
    model_path, vocab_path = train_or_load_sentencepiece(
        sampled_corpus_path,
        model_prefix=model_prefix,
        recipe=recipe,
        force_retrain=force_retrain_tokenizer,
    )
    processor = load_sentencepiece_processor(model_path)

    phase_token_info: dict[str, object] = {}
    for phase_name, phase_path in phase_file_map.items():
        phase_path = Path(phase_path)
        token_path = token_dir / f"{phase_name}_tokens.npy"
        token_path, token_count = encode_text_file_to_tokens(
            phase_path,
            token_path,
            processor=processor,
            eos_id=processor.eos_id(),
            force_rebuild=force_reencode,
        )
        phase_record: dict[str, object] = {
            "phase_name": phase_name,
            "phase_text_path": str(phase_path),
            "token_path": str(token_path),
            "token_count": int(token_count),
        }

        if phase_name in {"base", "adapt"}:
            train_path = token_dir / f"{phase_name}_train_tokens.npy"
            val_path = token_dir / f"{phase_name}_val_tokens.npy"
            phase_record["split"] = split_token_array(
                token_path,
                train_path=train_path,
                val_path=val_path,
                seq_len=seq_len,
                val_fraction=val_fraction,
                force_rebuild=force_reencode,
            )
        phase_token_info[phase_name] = phase_record

    manifest = {
        "tokenizer_recipe": asdict(recipe),
        "model_path": str(model_path),
        "vocab_path": str(vocab_path),
        "sampled_corpus_path": str(sampled_corpus_path),
        "vocab_size": int(processor.get_piece_size()),
        "pad_id": int(processor.pad_id()),
        "bos_id": int(processor.bos_id()),
        "eos_id": int(processor.eos_id()),
        "unk_id": int(processor.unk_id()),
        "phase_token_info": phase_token_info,
    }
    return manifest


def save_token_manifest(manifest: dict[str, object], path: Path | str) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return path

In [ ]:
# FineWeb Edu configuration and corpus artifact paths

corpus_recipe = CorpusRecipe()
fineweb_cache_path = CORPUS_ROOT / f"fineweb_edu_docs_{corpus_recipe.fineweb_edu.max_docs}.json"
corpus_manifest_path = CORPUS_ROOT / "corpus_manifest.json"
phase_dir = CORPUS_ROOT / "phase_text"
phase_dir.mkdir(exist_ok=True)
token_manifest_path = TOKENIZER_ROOT / "token_manifest.json"

print("FineWeb Edu recipe:")
print(json.dumps(recipe_to_jsonable(corpus_recipe)["fineweb_edu"], indent=2))
print("\nTokenizer recipe:")
print(json.dumps(recipe_to_jsonable(corpus_recipe)["tokenizer"], indent=2))
print("\nFineWeb cache path:", fineweb_cache_path)
print("Corpus manifest path:", corpus_manifest_path)
print("Token manifest path:", token_manifest_path)


In [ ]:
# Load FineWeb Edu and repo-local specialization data

fineweb_documents = []
if datasets is None:
    print("Install `datasets` before trying to stream FineWeb Edu.")
else:
    fineweb_documents = load_or_build_fineweb_edu(
        corpus_recipe.fineweb_edu,
        cache_path=fineweb_cache_path,
        force_refresh=False,
    )

repo_documents = build_repo_local_corpus(PROJECT_ROOT)

print(f"Loaded {len(fineweb_documents)} FineWeb Edu documents.")
print(f"Collected {len(repo_documents)} repo-local documents/chunks.")

print("\nFineWeb source summary:")
print(json.dumps(summarize_corpus_sources(fineweb_documents), indent=2))
print("\nRepo-local source summary:")
print(json.dumps(summarize_corpus_sources(repo_documents), indent=2))

for doc in (fineweb_documents[:2] + repo_documents[:4]):
    print("=" * 100)
    print(doc["id"], "|", doc["label"])
    print(doc["text"][:600])


In [ ]:
# Build tokenizer/base/adapt/SFT phase corpora and save a manifest

combined_documents = [*fineweb_documents, *repo_documents]
phase_group_map = {}
phase_line_map = {}
phase_file_map = {}

for phase in corpus_recipe.phases:
    grouped = build_phase_documents(combined_documents, phase)
    phase_group_map[phase.name] = grouped
    phase_lines = weighted_phase_lines(grouped, phase)
    phase_line_map[phase.name] = phase_lines

    phase_path = phase_dir / f"{phase.name}.txt"
    phase_path.write_text("\n".join(phase_lines) + "\n", encoding="utf-8")
    phase_file_map[phase.name] = phase_path

    print("=" * 100)
    print(phase.name)
    print(f"phase lines: {len(phase_lines)}")
    print(json.dumps({
        'broad_web': len(set(grouped['broad_web'])),
        'repo': len(set(grouped['repo'])),
        'science': len(set(grouped['science'])),
        'code': len(set(grouped['code'])),
        'reasoning': len(set(grouped['reasoning'])),
    }, indent=2))
    print('saved to:', phase_path)

contamination = {
    'tokenizer_vs_sft': contamination_report(phase_line_map['tokenizer'], phase_line_map['sft']),
    'base_vs_adapt': contamination_report(phase_line_map['base'], phase_line_map['adapt']),
    'adapt_vs_sft': contamination_report(phase_line_map['adapt'], phase_line_map['sft']),
}

corpus_manifest = build_corpus_manifest(
    broad_documents=fineweb_documents,
    repo_documents=repo_documents,
    recipe=corpus_recipe,
    phase_line_map=phase_line_map,
    phase_group_map=phase_group_map,
    contamination=contamination,
)
corpus_manifest['phase_files'] = {name: str(path) for name, path in phase_file_map.items()}
corpus_manifest['source_counts'] = {
    'fineweb_docs': len(fineweb_documents),
    'repo_docs': len(repo_documents),
    'combined_docs': len(combined_documents),
}
corpus_manifest_path.write_text(json.dumps(corpus_manifest, indent=2), encoding='utf-8')

print("\nContamination summary:")
print(json.dumps(contamination, indent=2))
print("\nSaved corpus manifest:", corpus_manifest_path)


In [ ]:
# Train/load the tokenizer and export all token arrays directly from the notebook

if sentencepiece is None:
    raise RuntimeError("Install `sentencepiece` in the notebook environment before building tokens.")

token_manifest = build_phase_token_artifacts(
    phase_file_map=phase_file_map,
    tokenizer_dir=TOKENIZER_ROOT,
    token_dir=PHASE_TOKEN_ROOT,
    recipe=corpus_recipe.tokenizer,
    seq_len=2048,
    val_fraction=0.02,
    force_retrain_tokenizer=False,
    force_reencode=False,
)
save_token_manifest(token_manifest, token_manifest_path)

print(json.dumps(token_manifest, indent=2)[:5000])

base_train = Path(token_manifest['phase_token_info']['base']['split']['train_path'])
base_val = Path(token_manifest['phase_token_info']['base']['split']['val_path'])
adapt_train = Path(token_manifest['phase_token_info']['adapt']['split']['train_path'])
adapt_val = Path(token_manifest['phase_token_info']['adapt']['split']['val_path'])

print("\nResolved training token paths:")
print('base_train :', base_train)
print('base_val   :', base_val)
print('adapt_train:', adapt_train)
print('adapt_val  :', adapt_val)


In [ ]:
# Model and architecture helpers

from __future__ import annotations

import math
from dataclasses import dataclass, field

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    vocab_size: int
    max_seq_len: int = 2048
    d_model: int = 768
    n_heads: int = 12
    n_kv_heads: int = 4
    n_layers: int = 12
    mlp_ratio: float = 4.0
    dropout: float = 0.0
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    rms_norm_eps: float = 1e-5
    rope_base: float = 10000.0
    tie_embeddings: bool = True
    qk_norm: bool = False
    sliding_window: int | None = None
    moe_num_experts: int = 0
    moe_top_k: int = 2
    moe_every_n_layers: int = 0
    moe_aux_loss_coef: float = 0.01
    init_std: float = 0.02


def count_parameters(module: nn.Module) -> int:
    return sum(param.numel() for param in module.parameters())


def active_parameter_count(cfg: ModelConfig) -> int:
    dense_params = cfg.d_model * cfg.vocab_size * (2 if not cfg.tie_embeddings else 1)
    mlp_hidden = int(cfg.d_model * cfg.mlp_ratio)
    attn_params = cfg.n_layers * (
        cfg.d_model * cfg.d_model * 2
        + cfg.d_model * (cfg.n_kv_heads * (cfg.d_model // cfg.n_heads)) * 2
    )
    mlp_params = cfg.n_layers * (3 * cfg.d_model * mlp_hidden)
    if cfg.moe_num_experts > 0:
        moe_layers = cfg.n_layers // max(cfg.moe_every_n_layers, 1)
        mlp_params += moe_layers * (cfg.moe_num_experts - 1) * (3 * cfg.d_model * mlp_hidden) // max(cfg.moe_num_experts, 1)
    return dense_params + attn_params + mlp_params


def estimate_kv_cache_bytes(cfg: ModelConfig, batch_size: int, seq_len: int, dtype_bytes: int = 2) -> int:
    head_dim = cfg.d_model // cfg.n_heads
    return batch_size * seq_len * cfg.n_layers * cfg.n_kv_heads * head_dim * 2 * dtype_bytes


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scale = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x * scale * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, base: float = 10000.0) -> None:
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._cached_seq_len = 0
        self._cached_cos: torch.Tensor | None = None
        self._cached_sin: torch.Tensor | None = None

    def _build_cache(self, seq_len: int, device: torch.device, dtype: torch.dtype) -> None:
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self._cached_cos = emb.cos().to(dtype=dtype)
        self._cached_sin = emb.sin().to(dtype=dtype)
        self._cached_seq_len = seq_len

    def forward(self, seq_len: int, device: torch.device, dtype: torch.dtype) -> tuple[torch.Tensor, torch.Tensor]:
        if self._cached_cos is None or seq_len > self._cached_seq_len or self._cached_cos.device != device or self._cached_cos.dtype != dtype:
            self._build_cache(seq_len, device, dtype)
        return self._cached_cos[:seq_len], self._cached_sin[:seq_len]


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x[..., ::2], x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    return (x * cos) + (rotate_half(x) * sin)


class SwiGLU(nn.Module):
    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        hidden = int(cfg.d_model * cfg.mlp_ratio)
        hidden = int(math.ceil(hidden / 256) * 256)
        self.up_proj = nn.Linear(cfg.d_model, hidden, bias=False)
        self.gate_proj = nn.Linear(cfg.d_model, hidden, bias=False)
        self.down_proj = nn.Linear(hidden, cfg.d_model, bias=False)
        self.dropout = nn.Dropout(cfg.resid_dropout)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, dict[str, float]]:
        gated = F.silu(self.gate_proj(x)) * self.up_proj(x)
        out = self.dropout(self.down_proj(gated))
        diag = {
            "mlp_hidden_norm": float(gated.detach().norm(dim=-1).mean().item()),
            "mlp_out_norm": float(out.detach().norm(dim=-1).mean().item()),
        }
        return out, diag


class MoEFeedForward(nn.Module):
    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.num_experts = cfg.moe_num_experts
        self.top_k = cfg.moe_top_k
        self.experts = nn.ModuleList([SwiGLU(cfg) for _ in range(self.num_experts)])
        self.router = nn.Linear(cfg.d_model, self.num_experts, bias=False)
        self.aux_loss_coef = cfg.moe_aux_loss_coef

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, dict[str, float]]:
        bsz, seq_len, dim = x.shape
        flat = x.reshape(bsz * seq_len, dim)
        router_logits = self.router(flat)
        router_probs = F.softmax(router_logits, dim=-1)
        topk_vals, topk_idx = torch.topk(router_probs, k=self.top_k, dim=-1)
        topk_vals = topk_vals / topk_vals.sum(dim=-1, keepdim=True)

        output = torch.zeros_like(flat)
        expert_usage = torch.zeros(self.num_experts, device=x.device, dtype=x.dtype)

        for expert_id, expert in enumerate(self.experts):
            mask = topk_idx == expert_id
            if not mask.any():
                continue
            token_ids, slot_ids = mask.nonzero(as_tuple=True)
            expert_input = flat[token_ids]
            expert_out, _ = expert(expert_input.unsqueeze(1))
            weights = topk_vals[token_ids, slot_ids].unsqueeze(-1)
            output[token_ids] += expert_out.squeeze(1) * weights
            expert_usage[expert_id] = mask.sum()

        router_mean = router_probs.mean(dim=0)
        usage_mean = expert_usage / max(float(expert_usage.sum().item()), 1.0)
        aux_loss = float((self.num_experts * torch.sum(router_mean * usage_mean)).item())
        entropy = float((-(router_probs.clamp_min(1e-9) * router_probs.clamp_min(1e-9).log()).sum(dim=-1).mean()).item())

        diag = {
            "router_aux_loss": aux_loss,
            "router_entropy": entropy,
            "expert_usage_max": float(usage_mean.max().item()) if expert_usage.sum() > 0 else 0.0,
            "expert_usage_min": float(usage_mean.min().item()) if expert_usage.sum() > 0 else 0.0,
            "expert_usage": usage_mean.detach().cpu().tolist() if expert_usage.sum() > 0 else [0.0] * self.num_experts,
        }
        return output.reshape(bsz, seq_len, dim), diag


class Attention(nn.Module):
    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        assert cfg.n_heads % cfg.n_kv_heads == 0
        self.cfg = cfg
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.q_norm = RMSNorm(self.head_dim, cfg.rms_norm_eps) if cfg.qk_norm else None
        self.k_norm = RMSNorm(self.head_dim, cfg.rms_norm_eps) if cfg.qk_norm else None
        self.attn_dropout = nn.Dropout(cfg.attn_dropout)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)
        self.rope = RotaryEmbedding(self.head_dim, cfg.rope_base)

    def _expand_kv(self, x: torch.Tensor) -> torch.Tensor:
        if self.n_heads == self.n_kv_heads:
            return x
        repeat = self.n_heads // self.n_kv_heads
        return x.repeat_interleave(repeat, dim=1)

    def forward(
        self,
        x: torch.Tensor,
        kv_cache: dict[str, torch.Tensor] | None = None,
        use_cache: bool = False,
        capture_diagnostics: bool = False,
    ) -> tuple[torch.Tensor, dict[str, float], dict[str, torch.Tensor] | None]:
        bsz, seq_len, _ = x.shape
        q = self.q_proj(x).view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(bsz, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(bsz, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if self.q_norm is not None:
            q = self.q_norm(q)
            k = self.k_norm(k)

        cache_len = 0 if kv_cache is None else kv_cache["k"].shape[-2]
        cos, sin = self.rope(cache_len + seq_len, x.device, q.dtype)
        q = apply_rope(q, cos[cache_len:], sin[cache_len:])
        k = apply_rope(k, cos[cache_len:], sin[cache_len:])

        if kv_cache is not None:
            k = torch.cat([kv_cache["k"], k], dim=-2)
            v = torch.cat([kv_cache["v"], v], dim=-2)
        next_cache = {"k": k, "v": v} if use_cache else None

        q_full = q
        k_full = self._expand_kv(k)
        v_full = self._expand_kv(v)
        scale = self.head_dim ** -0.5
        scores = torch.matmul(q_full, k_full.transpose(-1, -2)) * scale

        causal_offset = k_full.shape[-2] - q_full.shape[-2]
        causal_mask = torch.ones(q_full.shape[-2], k_full.shape[-2], device=x.device, dtype=torch.bool).tril(diagonal=causal_offset)
        if self.cfg.sliding_window is not None:
            window_mask = torch.zeros_like(causal_mask)
            for row in range(q_full.shape[-2]):
                left = max(0, k_full.shape[-2] - q_full.shape[-2] + row - self.cfg.sliding_window + 1)
                right = k_full.shape[-2] - q_full.shape[-2] + row + 1
                window_mask[row, left:right] = True
            causal_mask &= window_mask
        scores = scores.masked_fill(~causal_mask.view(1, 1, *causal_mask.shape), float("-inf"))

        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = torch.matmul(attn, v_full)
        out = out.transpose(1, 2).contiguous().view(bsz, seq_len, self.cfg.d_model)
        out = self.resid_dropout(self.out_proj(out))

        diag = {
            "attn_entropy": float((-(attn.clamp_min(1e-9) * attn.clamp_min(1e-9).log()).sum(dim=-1).mean()).item()),
            "q_norm": float(q.detach().norm(dim=-1).mean().item()),
            "k_norm": float(k.detach().norm(dim=-1).mean().item()),
            "v_norm": float(v.detach().norm(dim=-1).mean().item()),
        }
        if capture_diagnostics:
            diag["head_entropy_std"] = float((-(attn.clamp_min(1e-9) * attn.clamp_min(1e-9).log()).sum(dim=-1).mean(dim=(0, 2)).std()).item())
        return out, diag, next_cache


class TransformerBlock(nn.Module):
    def __init__(self, cfg: ModelConfig, layer_idx: int) -> None:
        super().__init__()
        self.layer_idx = layer_idx
        self.attn_norm = RMSNorm(cfg.d_model, cfg.rms_norm_eps)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.rms_norm_eps)
        self.attn = Attention(cfg)
        use_moe = cfg.moe_num_experts > 0 and cfg.moe_every_n_layers > 0 and ((layer_idx + 1) % cfg.moe_every_n_layers == 0)
        self.ffn = MoEFeedForward(cfg) if use_moe else SwiGLU(cfg)
        self.uses_moe = use_moe

    def forward(
        self,
        x: torch.Tensor,
        kv_cache: dict[str, torch.Tensor] | None = None,
        use_cache: bool = False,
        capture_diagnostics: bool = False,
    ) -> tuple[torch.Tensor, dict[str, float], dict[str, torch.Tensor] | None]:
        resid = x
        attn_out, attn_diag, next_cache = self.attn(self.attn_norm(x), kv_cache=kv_cache, use_cache=use_cache, capture_diagnostics=capture_diagnostics)
        x = resid + attn_out
        resid_after_attn = float(x.detach().norm(dim=-1).mean().item())
        ffn_out, ffn_diag = self.ffn(self.ffn_norm(x))
        x = x + ffn_out
        diag = {
            "residual_norm_post_attn": resid_after_attn,
            "residual_norm_post_ffn": float(x.detach().norm(dim=-1).mean().item()),
            **attn_diag,
            **ffn_diag,
        }
        return x, diag, next_cache


class ResearchTransformer(nn.Module):
    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.embed_tokens = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(cfg, layer_idx=i) for i in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model, cfg.rms_norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.embed_tokens.weight
        self.apply(self._init_weights)
        self._scale_residual_projections()

    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=self.cfg.init_std)
            if getattr(module, "bias", None) is not None:
                nn.init.zeros_(module.bias)

    def _scale_residual_projections(self) -> None:
        scale = 1.0 / math.sqrt(2 * self.cfg.n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=self.cfg.init_std * scale)
            if isinstance(block.ffn, SwiGLU):
                nn.init.normal_(block.ffn.down_proj.weight, mean=0.0, std=self.cfg.init_std * scale)
            else:
                for expert in block.ffn.experts:
                    nn.init.normal_(expert.down_proj.weight, mean=0.0, std=self.cfg.init_std * scale)

    def forward(
        self,
        input_ids: torch.Tensor,
        targets: torch.Tensor | None = None,
        use_cache: bool = False,
        past_key_values: list[dict[str, torch.Tensor] | None] | None = None,
        capture_diagnostics: bool = False,
    ) -> dict[str, object]:
        x = self.dropout(self.embed_tokens(input_ids))
        next_cache: list[dict[str, torch.Tensor] | None] = []
        diagnostics: list[dict[str, float]] = []

        for idx, block in enumerate(self.blocks):
            layer_cache = None if past_key_values is None else past_key_values[idx]
            x, block_diag, cache = block(x, kv_cache=layer_cache, use_cache=use_cache, capture_diagnostics=capture_diagnostics)
            diagnostics.append(block_diag)
            next_cache.append(cache)

        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))
        aux_loss = 0.0
        if diagnostics:
            aux_loss = sum(diag.get("router_aux_loss", 0.0) for diag in diagnostics) * self.cfg.moe_aux_loss_coef
        return {
            "logits": logits,
            "loss": loss,
            "aux_loss": loss.new_tensor(aux_loss) if loss is not None else torch.tensor(aux_loss, device=logits.device),
            "past_key_values": next_cache if use_cache else None,
            "diagnostics": diagnostics,
            "hidden_states": x,
        }

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 64,
        temperature: float = 0.8,
        top_k: int | None = 40,
    ) -> torch.Tensor:
        generated = input_ids
        past = None
        for _ in range(max_new_tokens):
            step_input = generated[:, -1:] if past is not None else generated[:, -self.cfg.max_seq_len :]
            out = self(step_input, use_cache=True, past_key_values=past)
            past = out["past_key_values"]
            logits = out["logits"][:, -1, :] / max(temperature, 1e-5)
            if top_k is not None:
                values, indices = torch.topk(logits, k=min(top_k, logits.shape[-1]), dim=-1)
                filtered = torch.full_like(logits, float("-inf"))
                filtered.scatter_(1, indices, values)
                logits = filtered
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            generated = torch.cat([generated, next_token], dim=-1)
        return generated

In [ ]:
# Dense and optional MoE model candidates

dense_cfg = ModelConfig(
    vocab_size=32000,
    max_seq_len=2048,
    d_model=768,
    n_heads=12,
    n_kv_heads=4,
    n_layers=18,
    mlp_ratio=3.5,
    dropout=0.0,
    attn_dropout=0.0,
    resid_dropout=0.0,
    qk_norm=True,
)

moe_cfg = ModelConfig(
    vocab_size=32000,
    max_seq_len=2048,
    d_model=768,
    n_heads=12,
    n_kv_heads=4,
    n_layers=18,
    mlp_ratio=3.5,
    dropout=0.0,
    attn_dropout=0.0,
    resid_dropout=0.0,
    qk_norm=True,
    moe_num_experts=4,
    moe_top_k=2,
    moe_every_n_layers=3,
)

summary_rows = []
for name, cfg in [('dense', dense_cfg), ('moe', moe_cfg)]:
    summary_rows.append({
        'name': name,
        'params_m': count_parameters(ResearchTransformer(cfg)) / 1e6,
        'kv_cache_mb@2k': estimate_kv_cache_bytes(cfg, batch_size=1, seq_len=2048) / 1024**2,
        'n_layers': cfg.n_layers,
        'n_heads': cfg.n_heads,
        'n_kv_heads': cfg.n_kv_heads,
        'moe_num_experts': cfg.moe_num_experts,
    })

print(json.dumps(summary_rows, indent=2))
fig = go.Figure()
fig.add_bar(name='params (M)', x=[row['name'] for row in summary_rows], y=[row['params_m'] for row in summary_rows])
fig.add_bar(name='KV cache MB @2k', x=[row['name'] for row in summary_rows], y=[row['kv_cache_mb@2k'] for row in summary_rows])
fig.update_layout(barmode='group', title='Dense vs optional MoE candidate scale', template='plotly_white')
fig.show()

In [ ]:
# Symbolic and tiny runtime smoke tests

L, T, H_kv, d_h, B = sp.symbols('L T H_kv d_h B', positive=True)
kv_bytes_expr = 2 * B * L * T * H_kv * d_h
print('KV-cache bytes expression:')
display(kv_bytes_expr)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for name, cfg in [('dense', dense_cfg), ('moe', moe_cfg)]:
    model = ResearchTransformer(cfg).to(device)
    x = torch.randint(0, cfg.vocab_size, (2, 128), device=device)
    y = torch.randint(0, cfg.vocab_size, (2, 128), device=device)
    out = model(x, targets=y, capture_diagnostics=True)
    print('=' * 100)
    print(name)
    print('logits shape:', tuple(out['logits'].shape))
    print('loss:', float(out['loss']))
    print('aux_loss:', float(out['aux_loss']))
    print('last block diagnostics:')
    print(json.dumps(out['diagnostics'][-1], indent=2))

In [ ]:
# Training and launch helpers

from __future__ import annotations

import json
import random
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import torch


@dataclass
class StageConfig:
    name: str
    stage_type: str
    train_tokens_path: str | None = None
    val_tokens_path: str | None = None
    max_steps: int = 2000
    learning_rate: float = 3e-4
    min_lr_scale: float = 0.1
    warmup_steps: int = 100
    grad_accum_steps: int = 8
    micro_batch_size: int = 2
    seq_len: int = 2048
    eval_interval: int = 50
    save_interval: int = 100
    notes: str = ""


@dataclass
class TrainConfig:
    seed: int = 42
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    beta1: float = 0.9
    beta2: float = 0.95
    use_bf16: bool = True
    activation_checkpointing: bool = True
    log_interval: int = 1
    eval_batches: int = 10
    run_name: str = "small_frontier_dense"
    stage_name: str = "base"
    extra_metadata: dict[str, object] = field(default_factory=dict)


@dataclass
class FsdpLaunchConfig:
    module: str = "transformer_lab.fsdp_worker"
    nproc_per_node: int = 1
    standalone: bool = True

    def as_command(self, bundle_path: Path | str) -> list[str]:
        command = ["torchrun"]
        if self.standalone:
            command.append("--standalone")
        command.extend(["--nproc_per_node", str(self.nproc_per_node), "-m", self.module, "--bundle", str(bundle_path)])
        return command


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cosine_with_warmup(step: int, max_steps: int, learning_rate: float, min_lr_scale: float, warmup_steps: int) -> float:
    if step < warmup_steps:
        return learning_rate * (step + 1) / max(warmup_steps, 1)
    progress = min(max((step - warmup_steps) / max(max_steps - warmup_steps, 1), 0.0), 1.0)
    cosine = 0.5 * (1.0 + np.cos(np.pi * progress))
    min_lr = learning_rate * min_lr_scale
    return float(min_lr + cosine * (learning_rate - min_lr))


def load_token_array(path: Path | str) -> np.ndarray:
    array = np.load(Path(path), mmap_mode="r")
    if array.ndim != 1:
        raise ValueError(f"Expected flat token array, got shape {array.shape}")
    return array


def sample_lm_batch(tokens: np.ndarray, batch_size: int, seq_len: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    starts = np.random.randint(0, len(tokens) - seq_len - 1, size=batch_size)
    x = np.stack([tokens[start : start + seq_len] for start in starts]).astype(np.int64)
    y = np.stack([tokens[start + 1 : start + seq_len + 1] for start in starts]).astype(np.int64)
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)


def build_stage_plan(
    base_train_path: str,
    base_val_path: str,
    adapt_train_path: str,
    adapt_val_path: str,
) -> list[StageConfig]:
    return [
        StageConfig(
            name="base_pretrain",
            stage_type="pretrain",
            train_tokens_path=base_train_path,
            val_tokens_path=base_val_path,
            max_steps=3000,
            learning_rate=3e-4,
            notes="Broad pretraining with repo-local structure kept in the mixture.",
        ),
        StageConfig(
            name="domain_adapt",
            stage_type="adapt",
            train_tokens_path=adapt_train_path,
            val_tokens_path=adapt_val_path,
            max_steps=1200,
            learning_rate=1.5e-4,
            notes="Domain adaptation on reasoning/science/systems-heavy text.",
        ),
        StageConfig(
            name="instruction_sft",
            stage_type="sft",
            max_steps=600,
            learning_rate=8e-5,
            notes="Supervised finetuning on curated instruction data.",
        ),
        StageConfig(
            name="reward_refine",
            stage_type="reward",
            max_steps=300,
            learning_rate=5e-5,
            notes="Lightweight verifier-guided refinement.",
        ),
    ]


def write_launch_bundle(
    path: Path | str,
    model_cfg,
    train_cfg: TrainConfig,
    stage_cfg: StageConfig,
) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "model_cfg": asdict(model_cfg),
        "train_cfg": asdict(train_cfg),
        "stage_cfg": asdict(stage_cfg),
    }
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return path

In [ ]:
# Stage plan and launch bundles

required_token_paths = [base_train, base_val, adapt_train, adapt_val]
missing = [path for path in required_token_paths if not Path(path).exists()]
if missing:
    raise RuntimeError(
        'Missing generated token files. Run the tokenizer/token-export cell first:\n' + '\n'.join(str(path) for path in missing)
    )

stage_plan = build_stage_plan(str(base_train), str(base_val), str(adapt_train), str(adapt_val))
for stage in stage_plan:
    print(stage)

base_train_cfg = TrainConfig(run_name='small_frontier_dense', stage_name=stage_plan[0].name)
moe_train_cfg = TrainConfig(run_name='small_frontier_moe', stage_name=stage_plan[0].name)

dense_bundle_path = write_launch_bundle(LAUNCH_ROOT / 'dense_base_bundle.json', dense_cfg, base_train_cfg, stage_plan[0])
moe_bundle_path = write_launch_bundle(LAUNCH_ROOT / 'moe_base_bundle.json', moe_cfg, moe_train_cfg, stage_plan[0])

print('Dense launch bundle:', dense_bundle_path)
print('MoE launch bundle:', moe_bundle_path)


In [ ]:
# Kick off training from the notebook by using the generated bundle paths

launch_dense = FsdpLaunchConfig(module='transformer_lab.fsdp_worker', nproc_per_node=3).as_command(dense_bundle_path)
launch_moe = FsdpLaunchConfig(module='transformer_lab.fsdp_worker', nproc_per_node=3).as_command(moe_bundle_path)

print('Dense launch command:')
print(' '.join(launch_dense))
print('\nMoE launch command:')
print(' '.join(launch_moe))


In [ ]:
# Dashboard helpers

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.ndimage import gaussian_filter1d


def load_metrics_rows(path: Path | str) -> list[dict]:
    path = Path(path)
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def _series(rows: list[dict], key: str, default: float | None = None) -> np.ndarray:
    values = []
    for row in rows:
        value = row.get(key, default)
        values.append(np.nan if value is None else value)
    return np.asarray(values, dtype=float)


def build_training_overview(rows: list[dict]) -> go.Figure:
    steps = _series(rows, "step")
    train = _series(rows, "train_loss")
    val = _series(rows, "val_loss")
    grad = _series(rows, "grad_norm")
    lr = _series(rows, "lr")
    toks = _series(rows, "tokens_per_sec")
    mem = _series(rows, "max_memory_gib", default=None)
    if np.isnan(mem).all():
        mem = _series(rows, "rank0_max_reserved_gib")

    fig = make_subplots(
        rows=3,
        cols=2,
        subplot_titles=(
            "Train / Val Loss",
            "Learning Rate",
            "Gradient Norm",
            "Tokens / sec",
            "Memory (GiB)",
            "Instability Gap",
        ),
    )
    fig.add_trace(go.Scatter(x=steps, y=train, mode="lines+markers", name="train_loss"), row=1, col=1)
    fig.add_trace(go.Scatter(x=steps, y=val, mode="lines+markers", name="val_loss"), row=1, col=1)
    fig.add_trace(go.Scatter(x=steps, y=lr, mode="lines", name="lr"), row=1, col=2)
    fig.add_trace(go.Scatter(x=steps, y=grad, mode="lines", name="grad_norm"), row=2, col=1)
    fig.add_trace(go.Scatter(x=steps, y=toks, mode="lines", name="tokens_per_sec"), row=2, col=2)
    fig.add_trace(go.Scatter(x=steps, y=mem, mode="lines", name="memory_gib"), row=3, col=1)
    fig.add_trace(go.Scatter(x=steps, y=val - train, mode="lines", name="val_minus_train"), row=3, col=2)
    fig.update_layout(height=900, width=1200, title="Transformer Training Overview", template="plotly_white")
    return fig


def build_moe_dashboard(rows: list[dict]) -> go.Figure:
    steps = _series(rows, "step")
    router_entropy = _series(rows, "router_entropy")
    load_max = _series(rows, "expert_usage_max")
    load_min = _series(rows, "expert_usage_min")
    aux = _series(rows, "router_aux_loss")

    fig = make_subplots(rows=2, cols=2, subplot_titles=("Router Entropy", "Expert Usage Spread", "Router Aux Loss", "Collapse Gap"))
    fig.add_trace(go.Scatter(x=steps, y=router_entropy, mode="lines", name="router_entropy"), row=1, col=1)
    fig.add_trace(go.Scatter(x=steps, y=load_max, mode="lines", name="usage_max"), row=1, col=2)
    fig.add_trace(go.Scatter(x=steps, y=load_min, mode="lines", name="usage_min"), row=1, col=2)
    fig.add_trace(go.Scatter(x=steps, y=aux, mode="lines", name="aux_loss"), row=2, col=1)
    fig.add_trace(go.Scatter(x=steps, y=load_max - load_min, mode="lines", name="collapse_gap"), row=2, col=2)
    fig.update_layout(height=700, width=1100, title="MoE Routing Dashboard", template="plotly_white")
    return fig


def build_attention_probe_figure(diagnostics: list[dict]) -> go.Figure:
    layers = np.arange(1, len(diagnostics) + 1)
    entropy = np.asarray([layer.get("attn_entropy", np.nan) for layer in diagnostics], dtype=float)
    q_norm = np.asarray([layer.get("q_norm", np.nan) for layer in diagnostics], dtype=float)
    k_norm = np.asarray([layer.get("k_norm", np.nan) for layer in diagnostics], dtype=float)
    residual = np.asarray([layer.get("residual_norm_post_ffn", np.nan) for layer in diagnostics], dtype=float)

    fig = make_subplots(rows=2, cols=2, subplot_titles=("Attention Entropy", "Q/K Norms", "Residual Norm", "Entropy Trend"))
    fig.add_trace(go.Scatter(x=layers, y=entropy, mode="lines+markers", name="attn_entropy"), row=1, col=1)
    fig.add_trace(go.Scatter(x=layers, y=q_norm, mode="lines+markers", name="q_norm"), row=1, col=2)
    fig.add_trace(go.Scatter(x=layers, y=k_norm, mode="lines+markers", name="k_norm"), row=1, col=2)
    fig.add_trace(go.Scatter(x=layers, y=residual, mode="lines+markers", name="residual_norm"), row=2, col=1)
    if len(entropy) > 2 and not np.isnan(entropy).all():
        smoothed = gaussian_filter1d(np.nan_to_num(entropy, nan=np.nanmean(entropy)), sigma=1)
        fig.add_trace(go.Scatter(x=layers, y=smoothed, mode="lines", name="entropy_smooth"), row=2, col=2)
    fig.update_layout(height=700, width=1100, title="Attention / Residual Probe", template="plotly_white")
    return fig


def build_next_token_figure(token_labels: list[str], probs: list[float]) -> go.Figure:
    probs_arr = np.asarray(probs, dtype=float)
    cumulative = np.cumsum(probs_arr)
    fig = make_subplots(rows=1, cols=3, subplot_titles=("Top-k Probabilities", "Cumulative Mass", "Distribution Histogram"))
    fig.add_trace(go.Bar(x=token_labels, y=probs_arr, name="top_k"), row=1, col=1)
    fig.add_trace(go.Scatter(x=np.arange(1, len(probs_arr) + 1), y=cumulative, mode="lines+markers", name="cumulative"), row=1, col=2)
    fig.add_trace(go.Histogram(x=probs_arr, nbinsx=20, name="hist"), row=1, col=3)
    fig.update_layout(height=420, width=1200, title="Next-Token Probability Dashboard", template="plotly_white")
    return fig


def build_dashboard_bundle(rows: list[dict]) -> dict[str, go.Figure]:
    bundle = {"training": build_training_overview(rows)}
    if any("router_entropy" in row for row in rows):
        bundle["moe"] = build_moe_dashboard(rows)
    return bundle

In [ ]:
# Metrics and dashboard loader

def candidate_metric_paths():
    paths = []
    for cfg_name in ['small_frontier_dense', 'small_frontier_moe']:
        for stage in ['base_pretrain', 'domain_adapt']:
            paths.append(ARTIFACT_ROOT / cfg_name / stage / 'metrics.jsonl')
    return paths

available = [path for path in candidate_metric_paths() if path.exists()]
print('Available metric files:')
for path in available:
    print('-', path)

In [ ]:
# Interactive Plotly dashboards when metrics exist

available_metric_files = [path for path in candidate_metric_paths() if path.exists()]
if not available_metric_files:
    print('No metrics files found yet. Run a stage first, then re-open this cell.')
else:
    figures = []
    titles = []
    for metric_path in available_metric_files[:2]:
        rows = load_metrics_rows(metric_path)
        if not rows:
            continue
        titles.append(metric_path.parent.name)
        figures.append(build_training_overview(rows))
        if any('router_entropy' in row for row in rows):
            titles.append(metric_path.parent.name + '_moe')
            figures.append(build_moe_dashboard(rows))

    children = [go.FigureWidget(fig) for fig in figures]
    tab = widgets.Tab(children=children)
    for idx, title in enumerate(titles):
        tab.set_title(idx, title[:18])
    display(tab)

In [ ]:
# Attention and next-token diagnostics on a probe model

probe_model = ResearchTransformer(dense_cfg).to(device)
probe_ids = torch.randint(0, dense_cfg.vocab_size, (1, 96), device=device)
probe_out = probe_model(probe_ids, capture_diagnostics=True)
probe_fig = build_attention_probe_figure(probe_out['diagnostics'])
probe_fig.show()

prompt_ids = torch.randint(0, dense_cfg.vocab_size, (1, 64), device=device)
probe_out = probe_model(prompt_ids, capture_diagnostics=False)
probs = torch.softmax(probe_out['logits'][0, -1], dim=-1)
values, indices = torch.topk(probs, k=15)
labels = [str(int(index.item())) for index in indices]
next_token_fig = build_next_token_figure(labels, values.detach().cpu().tolist())
next_token_fig.show()

In [ ]:
# Post-training helpers

from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path

import torch
import torch.nn.functional as F


@dataclass
class InstructionExample:
    prompt: str
    response: str
    source: str


@dataclass
class PreferenceExample:
    prompt: str
    chosen: str
    rejected: str
    source: str


@dataclass
class RewardExample:
    prompt: str
    candidate: str
    reward: float
    source: str


def build_repo_instruction_set(root: Path | str) -> list[InstructionExample]:
    root = Path(root)
    readme = (root / "README.md").read_text(encoding="utf-8") if (root / "README.md").exists() else ""
    agents = (root / "agents.md").read_text(encoding="utf-8") if (root / "agents.md").exists() else ""
    examples = [
        InstructionExample(
            prompt="Summarize the purpose of the mlworkshop repository in 4 sentences.",
            response=readme.split("\n", 8)[0:8] and " ".join(line.strip() for line in readme.splitlines()[0:8] if line.strip()),
            source="README.md",
        ),
        InstructionExample(
            prompt="Explain the notebook-first rule for this repository.",
            response="Notebooks remain the primary experimentation surface, while helper modules exist only to keep iteration, debugging, and diagnostics cleaner.",
            source="agents.md",
        ),
        InstructionExample(
            prompt="Describe how a strong small transformer should differ from a toy decoder-only model.",
            response="A strong student-scale model should improve data quality, backbone design, stability, diagnostics, and post-training rather than only increasing parameters.",
            source="transformer_refactor",
        ),
        InstructionExample(
            prompt="Give a checklist for resuming a distributed run safely.",
            response="Verify checkpoint integrity, config compatibility, optimizer state, data pointers, random seeds, and the last completed step before relaunching.",
            source="transformer_refactor",
        ),
    ]
    if agents:
        examples.append(
            InstructionExample(
                prompt="What broader mission should the repository support?",
                response="The work should support long-horizon mastery in scientific ML, transformer systems, and high-impact AI capacity building.",
                source="agents.md",
            )
        )
    return [example for example in examples if example.response]


def pack_instruction_batch(prompt_ids: list[list[int]], response_ids: list[list[int]], pad_id: int) -> tuple[torch.Tensor, torch.Tensor]:
    max_len = max(len(prompt) + len(response) for prompt, response in zip(prompt_ids, response_ids))
    input_rows = []
    target_rows = []
    for prompt, response in zip(prompt_ids, response_ids):
        sequence = prompt + response
        targets = [-100] * len(prompt) + response
        pad_len = max_len - len(sequence)
        input_rows.append(sequence + [pad_id] * pad_len)
        target_rows.append(targets + [-100] * pad_len)
    return torch.tensor(input_rows, dtype=torch.long), torch.tensor(target_rows, dtype=torch.long)


def sequence_logprob(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    log_probs = F.log_softmax(logits, dim=-1)
    gathered = torch.gather(log_probs, dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)
    return gathered.sum(dim=-1)


def dpo_loss(
    policy_chosen_logp: torch.Tensor,
    policy_rejected_logp: torch.Tensor,
    ref_chosen_logp: torch.Tensor,
    ref_rejected_logp: torch.Tensor,
    beta: float = 0.1,
) -> torch.Tensor:
    logits = beta * ((policy_chosen_logp - policy_rejected_logp) - (ref_chosen_logp - ref_rejected_logp))
    return -F.logsigmoid(logits).mean()


def outcome_reward(candidate: str, reference: str | None = None, required_keywords: list[str] | None = None) -> float:
    score = 0.0
    text = candidate.lower()
    if reference:
        ref = reference.lower()
        overlap = len(set(text.split()) & set(ref.split())) / max(len(set(ref.split())), 1)
        score += 0.6 * overlap
    if required_keywords:
        hits = sum(1 for keyword in required_keywords if keyword.lower() in text)
        score += 0.4 * hits / max(len(required_keywords), 1)
    return float(min(score, 1.0))

In [ ]:
# Eval helpers

from __future__ import annotations

import math
from dataclasses import asdict, dataclass

import numpy as np
import torch


@dataclass
class EvalCase:
    group: str
    prompt: str
    target: str | None = None
    verifier: str = "keyword"


def build_default_eval_suite() -> list[EvalCase]:
    return [
        EvalCase("reasoning", "Explain why RoPE helps extrapolation better than learned absolute position embeddings."),
        EvalCase("systems", "Describe the tradeoff between dense MLP blocks and sparse MoE blocks in a student-scale LM."),
        EvalCase("science", "Explain why notebook-first experimentation matters in a scientific ML research workflow."),
        EvalCase("instruction", "Write a short checklist for safely resuming distributed training after a crash."),
    ]


def token_cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    vocab = logits.shape[-1]
    return torch.nn.functional.cross_entropy(logits.reshape(-1, vocab), targets.reshape(-1), reduction="mean")


def perplexity_from_losses(losses: list[float]) -> float:
    if not losses:
        return float("nan")
    return float(math.exp(sum(losses) / len(losses)))


def calibration_bins(confidences: list[float], correct: list[float], num_bins: int = 10) -> list[dict[str, float]]:
    if not confidences:
        return []
    bins = np.linspace(0.0, 1.0, num_bins + 1)
    conf_arr = np.asarray(confidences)
    corr_arr = np.asarray(correct)
    records = []
    for left, right in zip(bins[:-1], bins[1:]):
        mask = (conf_arr >= left) & (conf_arr < right if right < 1.0 else conf_arr <= right)
        if not mask.any():
            continue
        records.append(
            {
                "bin_left": float(left),
                "bin_right": float(right),
                "mean_confidence": float(conf_arr[mask].mean()),
                "mean_accuracy": float(corr_arr[mask].mean()),
                "count": float(mask.sum()),
            }
        )
    return records


def summarize_eval_results(records: list[dict]) -> dict[str, object]:
    grouped = {}
    for record in records:
        grouped.setdefault(record["group"], []).append(record["score"])
    return {
        "group_scores": {group: float(np.mean(scores)) for group, scores in grouped.items()},
        "overall_score": float(np.mean([record["score"] for record in records])) if records else float("nan"),
        "records": records,
    }

In [ ]:
# Curated post-training utilities, reward demo, and calibration summary

instruction_examples = build_repo_instruction_set(PROJECT_ROOT)
print(f'Loaded {len(instruction_examples)} instruction examples.')
for example in instruction_examples[:4]:
    print('=' * 100)
    print('PROMPT:', example.prompt)
    print('RESPONSE:', example.response)
    print('SOURCE:', example.source)

reward_demo = outcome_reward(
    candidate='Grouped-query attention reduces KV-cache pressure because keys and values are shared across groups of query heads.',
    reference='Grouped-query attention reduces KV-cache growth by sharing keys and values across multiple query heads.',
    required_keywords=['kv-cache', 'shared', 'heads'],
)
print('\nReward demo:', reward_demo)

calibration = calibration_bins(
    confidences=[0.2, 0.35, 0.51, 0.68, 0.74, 0.82, 0.93],
    correct=[0, 0, 1, 1, 1, 0, 1],
    num_bins=5,
)
print(json.dumps(calibration, indent=2))


## Recommended Run Order

1. Run the setup/import cell and confirm `datasets` and `sentencepiece` are available.
2. Run the FineWeb Edu and repo-local corpus cells.
3. Run the phase corpus export cell.
4. Run the tokenizer/token-export cell to build the tokenizer and all `.npy` token files from the notebook.
5. Run the dense smoke-test and launch-bundle cells.
6. Launch dense base pretraining with the printed `torchrun` command.
7. Re-open the dashboard cells once `metrics.jsonl` exists.
8. Run domain adaptation after the dense base checkpoint is stable.
9. Use the SFT and reward-guided cells after the dense model has a clean baseline.
10. Only then run the MoE branch and compare against dense under matched active compute.